In [0]:
process_name='pricing_data_migration_landing'
process_run_log_path ='pricing_analytics.bronze.pipeline_control_logs'
Landing_Layer_Path="/Volumes/pricing_analytics/storageac/pricing_analytics/landing"

In [0]:
processing_file_date=str(spark.sql(f"""select coalesce(date(max(process_table_date_time))+1, date('2023-01-01')) as processing_file_date from {process_run_log_path} where process_name='{process_name}' and process_status='Completed' """).collect()[0]['processing_file_date'])

In [0]:
from datetime import datetime
formated_file_date=datetime.strptime(processing_file_date, '%Y-%d-%m').strftime('%d%m%Y')

In [0]:
HTTPWebsorceRootURL='https://retailpricing.blob.core.windows.net/'
HTTPWebSourceRootFileName=dbutils.widgets.get('Current_File_name')
HTTPWebSourceFileName=f"/PW_MW_DR_{formated_file_date}.csv"

HTTPWebSourceFullURL=HTTPWebsorceRootURL+HTTPWebSourceRootFileName+HTTPWebSourceFileName

In [0]:
Landing_Layer_Folder_Path=Landing_Layer_Path+'/'+HTTPWebSourceRootFileName+'/'+processing_file_date
Landing_Layer_full_path=Landing_Layer_Folder_Path+HTTPWebSourceFileName

In [0]:
import requests

response = requests.get(HTTPWebSourceFullURL)

if response.status_code != 200:
    raise Exception(f"File not available: {HTTPWebSourceFullURL}")

file_content = response.content

dbutils.fs.mkdirs(Landing_Layer_Folder_Path)

dbutils.fs.put(
    Landing_Layer_full_path,
    file_content.decode("utf-8"),
    overwrite=True
)

In [0]:
spark.sql(f"""insert into pricing_analytics.bronze.pipeline_control_logs values(
    '{process_name}',
    '{processing_file_date}',
    'Completed',
    current_timestamp()
)""")